# Student Performance - Model Training & Analysis

This notebook focuses on predicting student success (passing) using various machine learning models. We also explore dimensionality reduction techniques like PCA and t-SNE to visualize the data and evaluate their impact on model performance.

## Setup & Configuration
We use MLflow to track all our experiments, parameters, and metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Set MLflow tracking URI (using local sqlite db)
mlflow.set_tracking_uri("sqlite:///preprocessing/mlflow.db")
mlflow.set_experiment("Student_Performance_Analysis")

print("Setup complete. MLflow tracking to: ", mlflow.get_tracking_uri())

##  Data Loading
We load the cleaned dataset and prepare it for training. We'll drop target-related columns (`pass`, `G3`, `score`) to avoid data leakage.

In [ ]:
def load_data():
    df = pd.read_csv("data/student_clean.csv")
    X = df.drop(columns=['pass', 'G3', 'score'])
    y = df['pass']
    return X, y

X, y = load_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data loaded: {X.shape[0]} samples, {X.shape[1]} features.")
X.head()

##  Dimensionality Reduction
Let's see if we can visualize our high-dimensional data in 2D and compare model results with reduced dimensions.

In [ ]:
def plot_reduction(X, y, method="PCA"):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    reducer = PCA(n_components=2) if method == "PCA" else TSNE(n_components=2, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=embedding[:, 0], y=embedding[:, 1], hue=y, palette="viridis", alpha=0.7)
    plt.title(f"{method} Visualization")
    plt.show()

plot_reduction(X, y, "PCA")
plot_reduction(X, y, "t-SNE")

##  Systematic Model Training
We'll now run a series of experiments using different models and hyperparameters.

In [ ]:
def train_and_log(model_name, model, X_tr, X_te, y_tr, y_te, params):
    with mlflow.start_run(run_name=model_name):
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        
        # Calcul des métriques
        acc = accuracy_score(y_te, preds)
        prec = precision_score(y_te, preds, zero_division=0)
        rec = recall_score(y_te, preds, zero_division=0)
        f1 = f1_score(y_te, preds, zero_division=0)
        
        # MLflow logging (using machine names)
        mlflow.log_params(params)
        mlflow.log_metrics({"accuracy": acc, "precision": prec, "recall": rec, "f1_score": f1})
        mlflow.sklearn.log_model(model, "model")
        
        # AFFICHAGE TRÈS CLAIR DANS LE NOTEBOOK
        print("\n" + "*"*60)
        print(f"📊 RÉSULTATS POUR : {model_name}")
        print(f"⚙️ Config: {params}")
        print("-"*60)
        print(f"✅ ACCURACY          : {acc:.4%}")
        print(f"🎯 PRECISION         : {prec:.4%}")
        print(f"📢 RAPPEL (RECALL)   : {rec:.4%}")
        print(f"🏆 F1-SCORE          : {f1:.4%}")
        print("*"*60 + "\n")
        
        return {
            "Modèle": model_name, 
            "Params": str(params), 
            "Accuracy": acc, 
            "Precision": prec, 
            "Rappel (Recall)": rec, 
            "F1-Score": f1
        }

# Préparation des données scalées
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
all_results = []

print("🚀 Lancement des expérimentations...\n")

## 1. KNN
for k in [3, 7]:
    res = train_and_log("KNN", KNeighborsClassifier(n_neighbors=k), 
                        X_train_scaled, X_test_scaled, y_train, y_test, {"k": k})
    all_results.append(res)

## 2. SVM
for C in [0.1, 1.0]:
    res = train_and_log("SVM", SVC(C=C, probability=True), 
                        X_train_scaled, X_test_scaled, y_train, y_test, {"C": C})
    all_results.append(res)

## 3. Random Forest
for n in [50, 100]:
    res = train_and_log("Random Forest", RandomForestClassifier(n_estimators=n, random_state=42), 
                        X_train, X_test, y_train, y_test, {"n_estimators": n})
    all_results.append(res)

print("✨ Expérimentations terminées !")

## 📈 Comparaison des Performances
Voici le tableau récapitulatif complet ainsi qu'une visualisation graphique des métriques.

In [ ]:
# Création de la table récapitulative
summary_df = pd.DataFrame(all_results)

print("Tableau des scores (trié par F1-Score) :")
display(summary_df.sort_values(by="F1-Score", ascending=False))

# Visualisation graphique
plt.figure(figsize=(14, 8))
df_melted = summary_df.melt(id_vars=["Modèle", "Params"], 
                            value_vars=["Accuracy", "Precision", "Rappel (Recall)", "F1-Score"],
                            var_name="Métrique", value_name="Score")

sns.barplot(data=df_melted, x="Modèle", y="Score", hue="Métrique", palette="muted")
plt.title("Comparaison des modèles par métrique")
plt.ylim(0, 1.0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()